# Lab1.5 — Contor binar pe 4 biți controlat cu butoane

Acest task combină citirea butoanelor și controlul LED-urilor într-o aplicație simplă cu stare internă: un contor binar pe 4 biți.


## Obiective

- combinarea intrărilor și ieșirilor digitale;
- detectarea unei noi apăsări de buton;
- utilizarea operațiilor pe biți;
- implementarea unui contor modulo 16;
- introducerea unei temporizări simple pentru reducerea efectului de bounce.


## 1. Comenzi

- **BTN0** — incrementare;
- **BTN1** — decrementare;
- **BTN2** — reset la 0;
- **BTN3** — oprirea programului.

Cele patru LED-uri afișează valoarea contorului în binar.


In [ ]:
from pynq.overlays.base import BaseOverlay
import time

base = BaseOverlay("base.bit")

## 2. Funcții auxiliare

Funcția `read_buttons()` construiește o valoare pe 4 biți din starea butoanelor, iar `show_value()` afișează pe LED-uri o valoare între 0 și 15.


In [ ]:
def read_buttons():
    value = 0
    for i, button in enumerate(base.buttons):
        value |= (button.read() & 1) << i
    return value


def show_value(value):
    value &= 0xF
    for i, led in enumerate(base.leds):
        if value & (1 << i):
            led.on()
        else:
            led.off()

## 3. Detectarea unei noi apăsări

Dacă folosim direct starea butonului, un buton ținut apăsat ar genera aceeași comandă de mai multe ori.

Pentru a detecta doar tranziția **0 → 1**, comparăm starea curentă cu starea anterioară:

```python
pressed = current & ~previous
```


## 4. Contorul

Rulați programul și testați toate cele patru butoane.


In [ ]:
count = 0
previous = 0

show_value(count)
print(f"Counter = {count:2d} ({count:04b})")

while True:
    current = read_buttons()
    pressed = current & ~previous & 0xF

    if pressed & 0b0001:          # BTN0
        count = (count + 1) & 0xF

    if pressed & 0b0010:          # BTN1
        count = (count - 1) & 0xF

    if pressed & 0b0100:          # BTN2
        count = 0

    if pressed & 0b1000:          # BTN3
        print("Program oprit.")
        break

    if pressed:
        show_value(count)
        print(f"Counter = {count:2d} ({count:04b})")
        previous = current
        time.sleep(0.05)           # debounce simplu
    else:
        previous = current

    time.sleep(0.01)

for led in base.leds:
    led.off()

## 5. Verificare

Testați explicit următoarele situații:

- incrementare `14 → 15 → 0`;
- decrementare `1 → 0 → 15`;
- reset dintr-o valoare diferită de zero;
- menținerea BTN0 apăsat pentru aproximativ o secundă;
- oprirea programului cu BTN3.


## 6. Exercițiu

Modificați programul astfel încât **SW0** să selecteze pasul contorului:

- `SW0 = 0` → pas = 1;
- `SW0 = 1` → pas = 2.

Incrementarea și decrementarea trebuie să rămână modulo 16.


In [ ]:
# Scrieți aici modificarea


## 7. Întrebări

1. De ce se folosește expresia `pressed = current & ~previous`?
2. Ce rol are masca `& 0xF`?
3. De ce valoarea trece de la 15 la 0 la incrementare?
4. Ce problemă fizică încearcă să reducă întârzierea de 50 ms?
